# DSB: графики

Данные генерирует `setup.sh` → `dsb_drift.py`, ноутбук только читает результат:

```bash
SCALE=1 UPDATE_STREAMS=300 bash "DSB benchmark/setup.sh"
```

`rank.csv` — разброс внутри бакета по всем целочисленным колонкам фактовых таблиц,
`summary.csv` — метрики каждого чекпоинта, `<table>.<column>/step_XX/` — `freq`, `sample`, ворклоады.

In [ ]:
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

OUT = Path(os.environ.get("DSB_OUT", "../data/dsb/sf1"))
summary = pd.read_csv(OUT / "summary.csv")
RANK_CSV = OUT / "rank.csv" if (OUT / "rank.csv").exists() else OUT.parent / "sf1" / "rank.csv"
rank = pd.read_csv(RANK_CSV)
streams = pd.read_csv(OUT / "streams.csv")
COLUMNS = list(dict.fromkeys(summary["table"] + "." + summary["column"]))


def load_step(name, step):
    d = OUT / name / f"step_{step:02d}"
    z = np.load(d / "data.npz")
    return z["freq"], z["sample"], json.loads((d / "meta.json").read_text())


def buckets(freq, mn, mx, k):
    bw = int(np.ceil((mx - mn + 1) / k))
    idx = np.maximum(np.arange(len(freq)) - mn, 0) // bw
    return bw, np.bincount(idx, weights=freq)


COLUMNS

## 1. Где равномерность внутри бакета ломается сильнее всего

In [ ]:
top = rank.head(15).iloc[::-1]
labels = top["table"] + "." + top["column"]
fig, ax = plt.subplots(1, 2, figsize=(15, 6), sharey=True)
ax[0].barh(labels, top["uniform_qerr_p95"])
ax[0].set_xlabel("P95 q-error равномерного допущения (узкие запросы)")
ax[1].barh(labels, top["intra_bucket_cv"])
ax[1].set_xlabel("CV частот внутри бакета (взвешено по массе)")
plt.tight_layout()
rank.head(15)

## 2. Распределение и бакеты одной колонки

In [ ]:
NAME = COLUMNS[0]
freq, sample, meta = load_step(NAME, 0)
bw, cnt = buckets(freq, meta["mn"], meta["mx"], meta["k"])
plt.figure(figsize=(15, 4))
plt.bar(np.arange(len(cnt)), cnt, width=1.0, edgecolor="black", linewidth=0.2)
plt.title(f"{NAME}: {meta['k']} бакетов по {bw} значений, N = {meta['N']:,}")
plt.xlabel("бакет")
plt.ylabel("строк")
plt.show()

In [ ]:
seg = np.zeros(len(cnt) * bw)
seg[: len(freq) - meta["mn"]] = freq[meta["mn"]:]
per_bucket = seg.reshape(len(cnt), bw)
mean = per_bucket.mean(1)
cv = per_bucket.std(1) / np.where(mean > 0, mean, 1)
BUCKET = int(np.argmax(cv * (cnt > np.median(cnt))))
lo = meta["mn"] + BUCKET * bw
plt.figure(figsize=(15, 4))
plt.bar(np.arange(lo, lo + bw), per_bucket[BUCKET], width=1.0)
plt.axhline(mean[BUCKET], color="red", label="равномерное допущение")
plt.title(f"{NAME}: бакет {BUCKET} [{lo}, {lo + bw - 1}], CV = {cv[BUCKET]:.2f}")
plt.xlabel("значение")
plt.ylabel("строк")
plt.legend()
plt.show()

## 3. Насколько сильный дрейф

In [ ]:
fig, ax = plt.subplots(1, 4, figsize=(20, 4.5))
for name, g in summary.assign(name=summary["table"] + "." + summary["column"]).groupby("name", sort=False):
    ax[0].plot(g["threshold_pct"], g["ks_vs_step0"], marker=".", label=name)
    ax[1].plot(g["threshold_pct"], g["tv_vs_step0"], marker=".")
    ax[2].plot(g["threshold_pct"], g["stale_equiwidth_qerr_p95"], marker=".")
    ax[3].plot(g["threshold_pct"], g["N"] / g["N"].iloc[0], marker=".")
for a, t in zip(ax, ["KS к шагу 0", "TV на сетке шага 0", "P95 устаревшей Equi-Width", "N / N шага 0"]):
    a.set_title(t)
    a.set_xlabel("churn, %")
ax[0].legend(fontsize=7)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
for c in [c for c in streams.columns if c.endswith("_churn_pct")]:
    ax.plot(streams["stream"], streams[c], label=c.replace("_churn_pct", ""))
ax.axhline(100, color="grey", lw=0.5)
ax.set_xlabel("поток обновлений")
ax.set_ylabel("накопленный churn, %")
ax.legend()
plt.show()

## 4. Эволюция распределения по чекпоинтам

In [ ]:
def plot_evolution(name):
    steps = sorted(summary.loc[summary["table"] + "." + summary["column"] == name, "step"])
    f0, _, m0 = load_step(name, 0)
    bw0, base = buckets(f0, m0["mn"], m0["mx"], m0["k"])
    hist, labels = [], []
    for s in steps:
        f, _, m = load_step(name, s)
        _, h = buckets(f, m0["mn"], m0["mx"], m0["k"])
        h = np.pad(h, (0, max(0, len(base) - len(h))))[: len(base)]
        hist.append(h / h.sum())
        labels.append(f"{m['threshold_pct']}%")
    hist = np.array(hist)
    colors = plt.cm.viridis(np.linspace(0, 1, len(steps)))
    fig, ax = plt.subplots(1, 2, figsize=(18, 5))
    for h, c, lab in zip(hist, colors, labels):
        ax[0].plot(h, color=c, lw=0.8, label=lab)
    ax[0].set_title(f"{name}: доля строк по бакетам шага 0")
    ax[0].legend(fontsize=6, ncol=3)
    delta = (hist - hist[0]) / (hist[0] + 1e-12)
    im = ax[1].imshow(delta, aspect="auto", cmap="RdBu_r", vmin=-0.5, vmax=0.5)
    ax[1].set_yticks(range(len(labels)))
    ax[1].set_yticklabels(labels, fontsize=7)
    ax[1].set_xlabel("бакет")
    ax[1].set_title("относительное изменение к шагу 0")
    plt.colorbar(im, ax=ax[1])
    plt.tight_layout()
    plt.show()


for name in COLUMNS:
    plot_evolution(name)